In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

In [ ]:
HOME_PATH = '/content/drive/My Drive/AI/german_name_prediction'
DATA_PATH = os.path.join(HOME_PATH, 'data')

WEIGHT_PATH = os.path.join(HOME_PATH, 'weight')
if not os.path.isdir(WEIGHT_PATH):
    os.makedirs(WEIGHT_PATH)

## Preprocess data

## Load data

In [ ]:
with open(os.path.join(DATA_PATH, 'last_name.txt')) as f:
    data = f.read().split('\n')[:-1]

## Tokenize

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer = Tokenizer(char_level=True, filters='', split='', oov_token='*')

In [ ]:
pad_token = 0
unknow_token = 1

In [ ]:
tokenizer.fit_on_texts(data)

In [ ]:
seq_all = tokenizer.texts_to_sequences(data)

In [ ]:
len_all = [len(x) for x in seq_all]
max_len = max(len_all)

In [ ]:
seq_all = pad_sequences(seq_all, maxlen=max_len)

In [ ]:
X_all = seq_all[:,:-1]
y_all = seq_all[:,1:]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

In [ ]:
import torch
torch.save(tokenizer, os.path.join(DATA_PATH, 'last_name_tokenizer.h5'))

## Data loader

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset, ConcatDataset

In [ ]:
def build_dataset_from_tensors(X, y):
    ds = TensorDataset(X, y)
    return ds

In [ ]:
X_train = torch.tensor(X_train).long()
y_train = torch.tensor(y_train).long()
X_val = torch.tensor(X_val).long()
y_val = torch.tensor(y_val).long()

In [ ]:
train_ds = build_dataset_from_tensors(X_train, y_train)
val_ds = build_dataset_from_tensors(X_val, y_val)

In [ ]:
batch_size = 256
shuffle = True

In [ ]:
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=shuffle)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=shuffle)

## Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from tqdm import tqdm

In [ ]:
class Model(nn.Module):
    def __init__(self, vocab_size, embeding_size, hidden_size):
        super(Model, self).__init__()
        self.embeding = nn.Embedding(vocab_size, embeding_size)
        self.lstm = nn.LSTM(embeding_size, hidden_size, num_layers=2, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden_state=None):
        # x: BxS
        x = self.embeding(x) # BxSxE
        if hidden_state is None:
            x, hidden_state = self.lstm(x) # BxSx2H
        else:
            x, hidden_state = self.lstm(x, hidden_state) # BxSx2H
        x = F.relu(x)
        x = self.linear(x) # BxSxV
        return x, hidden_state

    def predict(self, x, hidden_state=None):
        x, hidden_state = self.forward(x, hidden_state)
        x = F.softmax(x, dim=-1) # BxSxV
        return x, hidden_state

In [ ]:
def forward_and_loss(model, x, y, loss_fn, pad_token):
    out, hidden_state = model(x)
    loss = loss_fn(out.view(-1, out.size(-1)), y.view(-1), ignore_index=pad_token)
    return out, loss

In [ ]:
def train_model(model, optim, train_iter, loss_fn, pad_token, weight_path=None, device=None):
    total_loss = 0.0

    model.train()

    with tqdm(total=len(train_iter)) as pbar:
        for x, y in train_iter:
            if device is not None and device.type=='cuda':
                x = x.cuda()
                y = y.cuda()

            optimizer.zero_grad()
            _, loss = forward_and_loss(model, x, y, loss_fn, pad_token=pad_token)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            pbar.update(1)
            pbar.set_description("%-10s = %.6f  " % ('loss', total_loss))

    # Save model
    if weight_path is not None:
        state = {
            "model": model.state_dict(),
            "optim": optimizer.state_dict()
        }

        torch.save(state, weight_path)

    return total_loss

In [ ]:
def evaluate_model(model, val_iter, pad_token, device=None):
    model.eval()
    with torch.no_grad(), tqdm(total=len(val_iter)) as pbar:
        total_loss = 0.0

        for x, y in val_iter:
            if device is not None and device.type=='cuda':
                x = x.cuda()
                y = y.cuda()

            _, loss = forward_and_loss(model, x, y, F.cross_entropy, pad_token=pad_token)

            total_loss += loss.item()

            pbar.update(1)
            pbar.set_description("%-10s = %.6f  " % ('val_loss', total_loss))

    return total_loss

## Training

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
embedding_size = 200
hidden_size = 256
learning_rate = 0.0001
loss_fn = F.cross_entropy

In [ ]:
device = torch.torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = Model(vocab_size, embedding_size, hidden_size)

In [ ]:
if device.type=='cuda':
    model = model.cuda()

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

In [ ]:
WEIGHT_PATH = os.path.join(HOME_PATH, 'ln_weight')
if not os.path.isdir(WEIGHT_PATH):
    os.makedirs(WEIGHT_PATH)

In [ ]:
# logging.info('\n' + '*'*50 + '\n' + "Start logging\nvocab_size=%d\nembedding_size=%d\nhidden_size=%d\nlearning_rate=%d\nbatch_size=%d\n" \
#              % (vocab_size, embedding_size, hidden_size, learning_rate, batch_size))

In [ ]:
num_epoch = 300

In [ ]:
for i in range(1, num_epoch+1):
    weight_path = None
    if i%10==0:
        weight_path = os.path.join(WEIGHT_PATH, 'epoch_%02d.h5' % i)
    print("\nEpoch %02d" % i, flush=True)
    train_loss = train_model(model, optimizer, train_dl, loss_fn, pad_token, weight_path, device)
    val_loss = evaluate_model(model, val_dl, pad_token, device)


Epoch 01


val_loss   = 1897.266603  : 100%|██████████| 843/843 [00:07<00:00, 113.35it/s]


Epoch 02



val_loss   = 1806.532337  : 100%|██████████| 843/843 [00:07<00:00, 112.83it/s]



Epoch 03


val_loss   = 1763.228501  : 100%|██████████| 843/843 [00:07<00:00, 112.25it/s]

... (21 outputs left out)


val_loss   = 1637.304709  : 100%|██████████| 843/843 [00:07<00:00, 114.39it/s]



Epoch 15


loss       = 2545.904263  :  39%|███▉      | 1306/3370 [00:31<00:50, 41.09it/s]

Buffered data was truncated after reaching the output size limit.